# Langfuse Dataset Upload — DiabetesAssist

Builds a **correctness dataset** from ground-truth Q&A pairs and uploads it to
Langfuse so every future eval run scores against the same gold standard.

Each dataset item has:
- `input` — the question sent to the RAG system
- `expected_output` — the reference (gold) answer

**Required `.env` keys:**
```
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_HOST=https://cloud.langfuse.com   # optional, defaults to cloud
```

In [ ]:
import sys, os
sys.path.append('..')

from dotenv import load_dotenv
load_dotenv('../.env')

from langfuse import Langfuse

lf = Langfuse(
    public_key=os.environ['LANGFUSE_PUBLIC_KEY'],
    secret_key=os.environ['LANGFUSE_SECRET_KEY'],
    host=os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com'),
)
print('Langfuse client ready')

## 1. Define Ground-Truth Q&A Pairs

These are clinically reviewed reference answers drawn from the ADA Standards of Care
and the documents ingested into the vectorstore. Add or edit rows here to expand
the dataset over time.

In [ ]:
import pandas as pd

df = pd.read_json('langfuse_dataset_medical-rag-eval.json')

CORRECTNESS_PAIRS = df.to_dict(orient='records')

print(f'{len(CORRECTNESS_PAIRS)} examples loaded')
df.head()

## 2. Create Dataset in Langfuse

`create_dataset` is idempotent — if the dataset name already exists Langfuse
returns the existing one rather than raising an error.

In [ ]:
DATASET_NAME = 'medical-rag-correctness'

lf.create_dataset(
    name=DATASET_NAME,
    description='Ground-truth Q&A pairs for DiabetesAssist RAG correctness evaluation',
    metadata={'source': 'ADA Standards of Care / ingested PDFs'},
)
print(f"Dataset '{DATASET_NAME}' ready")

## 3. Upload Items

Each call to `create_dataset_item` adds one Q&A pair. Items are deduplicated by
`id` if you pass one — useful for re-running without duplicating rows.

In [ ]:
for item in CORRECTNESS_PAIRS:
    lf.create_dataset_item(
        dataset_name=DATASET_NAME,
        input=item['input'],
        expected_output=item['expected_output'],
    )

lf.flush()
print(f'Uploaded {len(CORRECTNESS_PAIRS)} items to "{DATASET_NAME}"')

## 4. Verify Upload

Fetch items back from Langfuse to confirm they landed.

In [ ]:
dataset = lf.get_dataset(DATASET_NAME)
print(f'{len(dataset.items)} items in "{DATASET_NAME}":\n')
for item in dataset.items:
    print(f'  [{item.id}]')
    print(f'    input : {item.input["question"][:70]}')
    print(f'    output: {item.expected_output["answer"][:70]}\n')